# Basic Statistics

+ Harald Puhr
+ 2025-11-03

[Data Source](https://github.com/ha-pu/data_files/tree/main#user-knowledge-modeling-dataset)

## Load libraries and data

In [ ]:
import numpy as np
import pandas as pd
from plotnine import ggplot, aes, geom_boxplot, geom_histogram, geom_tile, facet_wrap, labs, theme_bw
from scipy import stats
from scipy.stats import ttest_ind, t

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/ha-pu/data_files/refs/heads/main/1-user_knowledge.csv")

Code variable `UNS` as type factor to better handle categorical data.

In [ ]:
df["UNS"] = pd.Categorical(
    df["UNS"],
    categories=["Very Low", "Low", "Middle", "High"],
    ordered=True
)

## Inspect data frame

In [ ]:
df.head()

## Get frequency of skill levels

In [ ]:
(
    df.groupby("UNS", observed=False)
        .size()
        .reset_index(name="n")
        .sort_values("UNS")
        .reset_index(drop=True)
)

## Analyze study effort by skill level

In [ ]:
(
    df.groupby("UNS", observed=False)
      .agg(
          n=("UNS", "size"),
          STG=("STG", "mean"),
          SCG=("SCG", "mean"),
          STR=("STR", "mean"),
          LPR=("LPR", "mean"),
          PEG=("PEG", "mean"),
      )
      .reset_index()
      .sort_values("UNS")
      .reset_index(drop=True)
)

### Questions

+ Are there immediate takeaways for our efforts to increase skills (UNS)?
+ Is there one effort-related variables that stands out?

## Distribution of study effort

In [ ]:
df_long = pd.melt(
    df,
    id_vars=[c for c in df.columns if c not in ["STG", "SCG", "STR", "LPR", "PEG"]],
    value_vars=["STG", "SCG", "STR", "LPR", "PEG"],
    var_name="knowledge",
    value_name="value"
)
df_long.head()

In [ ]:
p = (
    ggplot(df_long)
    + geom_histogram(aes("value"), bins=25)
    + facet_wrap("~knowledge")
    + theme_bw()
)
p

### Questions

+ What do you see in the distribution of study effort variables?
+ How do the distributions differentiate?
+ Are there immediate takeaways for our efforts to increase skills (UNS)?

## Boxplot of study effort by skill level

In [ ]:
p = (
    ggplot(df_long)
    + geom_boxplot(aes(x="UNS", y="value"))
    + facet_wrap("~knowledge")
    + theme_bw()
)
p

### Questions

+ Do the box plots tell us anything about the relevance of the study effort
  variables?
+ Are there immediate takeaways for our efforts to increase skills (UNS)?

## Correlation between study effort variables

In [ ]:
corr = df.drop(columns=["UNS"]).corr(method="pearson", numeric_only=True)
corr.head()

In [ ]:
corr_long = (
    corr.reset_index()
        .rename(columns={"index": "Var1"})
        .melt(id_vars="Var1", var_name="Var2", value_name="Correlation")
)
corr_long.head()

In [ ]:
p = (
    ggplot(corr_long)
    + geom_tile(aes(x="Var1", y="Var2", fill="Correlation"))
    + labs(x=None, y=None)
    + theme_bw()
)
p

### Questions

+ Are there any interesting correlations between study effort variables?
+ Are there immediate takeaways for our efforts to increase skills (UNS)?

## Test the statistical signficance of differences between skill levels

In [ ]:
df = df.assign(
    upper_half=df["UNS"].isin(["High", "Middle"]),
    top=df["UNS"].eq("High")
)
df.head()

In [ ]:
# samples
x = df.loc[df["upper_half"], "STG"].astype(float)
y = df.loc[~df["upper_half"], "STG"].astype(float)

# Welch t-test (matches R's t.test default)
t_stat, p_value = ttest_ind(x, y, equal_var=False, nan_policy="omit")
pd.DataFrame({"t_stat": t_stat, "p_value_two_sided": p_value}, index = [0])

In [ ]:
# Add Welch-Satterthwaite dof and a 95% CI for the mean difference
n1, n2 = x.count(), y.count()
s1, s2 = x.var(ddof=1), y.var(ddof=1)
mean_diff = x.mean() - y.mean()
se2 = s1/n1 + s2/n2
se = np.sqrt(se2)
df_welch = se2**2 / ((s1**2)/((n1**2)*(n1-1)) + (s2**2)/((n2**2)*(n2-1)))

alpha = 0.05
crit = t.ppf(1 - alpha/2, df_welch)
ci_low, ci_high = mean_diff - crit*se, mean_diff + crit*se

pd.DataFrame({
    "mean_x": x.mean(),
    "mean_y": y.mean(),
    "mean_diff": mean_diff,
    "ci_95": (ci_low, ci_high)
})

In [ ]:
# samples
x = df.loc[df["top"], "STG"].astype(float)
y = df.loc[~df["top"], "STG"].astype(float)

# Welch t-test (matches R's t.test default)
t_stat, p_value = ttest_ind(x, y, equal_var=False, nan_policy="omit")
pd.DataFrame({"t_stat": t_stat, "p_value_two_sided": p_value}, index = [0])

In [ ]:
# Add Welch-Satterthwaite dof and a 95% CI for the mean difference
n1, n2 = x.count(), y.count()
s1, s2 = x.var(ddof=1), y.var(ddof=1)
mean_diff = x.mean() - y.mean()
se2 = s1/n1 + s2/n2
se = np.sqrt(se2)
df_welch = se2**2 / ((s1**2)/((n1**2)*(n1-1)) + (s2**2)/((n2**2)*(n2-1)))

alpha = 0.05
crit = t.ppf(1 - alpha/2, df_welch)
ci_low, ci_high = mean_diff - crit*se, mean_diff + crit*se

pd.DataFrame({
    "mean_x": x.mean(),
    "mean_y": y.mean(),
    "mean_diff": mean_diff,
    "ci_95": (ci_low, ci_high)
})

In [ ]:
# One-way ANOVA: STG ~ UNS
# SciPy quick result (F and p-value)
groups = [g["STG"].dropna().to_numpy()
          for _, g in df.groupby("UNS", observed=False)]
F_scipy, p_scipy = stats.f_oneway(*groups)

pd.DataFrame({"F": F_scipy, "p_value": p_scipy}, index = [0])

### Questions

+ How are these results different from the previous ones?
+ What do you think about the results of the t-test and ANOVA?
+ Are there immediate takeaways for our efforts to increase skills (UNS)?
+ What analyses should we do next?
